In [0]:
import requests
import os
from datetime import datetime
import pandas as pd
from functools import reduce
from pyspark.sql import DataFrame

In [0]:
# --- CONFIGURAÇÕES DE INFRAESTRUTURA (PATHS) ---

PATH_RAW = f"/Volumes/lakehouse/raw/prices_diesel"

URL_BASE = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan"

FILE_PATTERNS = {
    2025: "precos-diesel-gnv-{mes:02d}.csv",
    2026: "{mes:02d}-dados-abertos-precos-diesel-gnv.csv"
}

In [0]:
def download_file(url, filename):
    try:
        response = requests.get(url, timeout=25)
        response.raise_for_status() 
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Sucesso: {filename}")
    except Exception as e:
        print(f"Arquivo não baixado {filename}: {e}")

for ano, formato in FILE_PATTERNS.items():
    print(f"\nIniciando downloads de {ano}...")
    mes_limite = datetime.now().month if ano == datetime.now().year else 13
    for mes in range(1, mes_limite):
        name_archive = formato.format(mes=mes)
        url_end = f"{URL_BASE}/{ano}/{name_archive}"
        
        os.makedirs(os.path.join(PATH_RAW, str(ano)), exist_ok=True)
        save_path = os.path.join(PATH_RAW, str(ano), name_archive)
        
        download_file(url_end, save_path)

In [0]:
paths = [f"{PATH_RAW}/{ano}/" for ano in FILE_PATTERNS]

df_final = spark.read.csv(paths, sep=";", header=True, inferSchema=True)

df_final = df_final.toDF(*[
    c.replace("-", "").replace("  ", " ").replace(" ", "_").lower() 
    for c in df_final.columns
])

df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lakehouse.bronze.precos_combustiveis")

In [0]:
%sql
SELECT * FROM lakehouse.bronze.precos_combustiveis